# 🧠 Tripolar EEG — Single Subject Analysis

Run this notebook to analyze **one subject** at a time. It loads the raw BrainVision files, performs full spectral decomposition, and generates all per-channel figures.

For group-level analysis across all subjects, see `group_analysis.ipynb`.

In [ ]:
import sys
sys.path.insert(0, '.')
from eeg_analysis import *

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print("Ready ✓")

## 1. Discover Available Subjects

In [ ]:
DATA_DIR = 'data/'

subjects = discover_subjects(DATA_DIR)
print(f"Found {len(subjects)} subjects:\n")
for s in subjects:
    avg_status = "✓" if s['avg'] else "✗"
    vmrk_status = "✓" if s['vmrk'] else "✗"
    print(f"  {s['name']:<15s}  files: {s['basename']}")
    print(f"  {'':15s}  markers: {vmrk_status}   avg: {avg_status}")
    print()

## 2. Load a Subject

Change the `SUBJECT` variable below to analyze a different subject.

In [ ]:
# ═══════════════════════════════════════
# CHANGE THIS to select your subject
# ═══════════════════════════════════════
SUBJECT = subjects[0]['name']  # or use a specific name like 'SK1'

print(f"Loading: {SUBJECT}")
subject = load_subject(DATA_DIR, SUBJECT)
print(f"  EEG shape:     {subject['eeg'].shape}")
print(f"  Duration:      {subject['n_samples']/FS:.1f}s ({subject['n_samples']/FS/60:.1f} min)")
print(f"  Stim triggers: {len(subject['stim_samples'])}")
print(f"  Stim blocks:   {len(subject['stim_blocks'])}")
print(f"  Open/close:    {len(subject['events_oc'])} markers → {len(subject['epochs_oc'])} epochs")
print(f"  Averaged VEP:  {'Yes (n=' + str(subject['avg_n_segments']) + ')' if subject['avg_data'] is not None else 'Not available'}")

if subject['epochs_oc']:
    print(f"\nEpochs:")
    for ep in subject['epochs_oc']:
        print(f"  {ep['label']:>5s}: {ep['start_s']:6.1f}s – {ep['end_s']:6.1f}s  ({ep['dur']:.1f}s)")

## 3. Channel Statistics

In [ ]:
eeg = subject['eeg']
print(f"{'Channel':<30} {'Min(µV)':>9} {'Max(µV)':>9} {'Std(µV)':>9} {'Clipping?':>10}")
print("═" * 72)
for i in range(N_CHANNELS):
    clips = "⚠ YES" if eeg[i].max() >= 3276.6 or eeg[i].min() <= -3276.6 else ""
    print(f"{CH_LABELS[i]:<30} {eeg[i].min():>9.1f} {eeg[i].max():>9.1f} "
          f"{eeg[i].std():>9.1f} {clips:>10}")

## 4. Run Analysis

In [ ]:
results = analyze_subject(subject)
print("Analysis complete ✓")
print(f"\n{'Channel':<30} {'αSNR(dB)':>9} {'αReact':>8} {'CorrDisc':>9} {'VEP P2P':>9}")
print("═" * 70)
for i in range(N_CHANNELS):
    print(f"{CH_LABELS[i]:<30} {results['alpha_snr'][i]:>8.2f}  "
          f"{results['alpha_reactivity'][i]:>7.2f}x {results['disc_correlation'][i]:>8.4f}  "
          f"{results['vep_p2p'][i]:>8.1f}")

## 5. Summary Dashboard

In [ ]:
plot_subject_summary(subject, results)

## 6. Raw Time Series with Events

In [ ]:
fig, axes = plt.subplots(11, 1, figsize=(18, 22), sharex=True)
fig.suptitle(f'Raw EEG (µV) — {subject["name"]}', fontsize=14, fontweight='bold', y=1.0)
ds = 10
for i in range(N_CHANNELS):
    ax = axes[i]
    ax.plot(subject['t'][::ds], eeg[i, ::ds], lw=0.3, color='#2c3e50')
    ax.set_ylabel(f'Ch{i+1}', fontsize=9)
    ymax = np.percentile(np.abs(eeg[i]), 99.5)
    ax.set_ylim(-ymax, ymax); ax.tick_params(labelsize=8)
    add_event_markers(ax, subject['events_oc'], subject['stim_blocks'], subject['close_epochs'])
axes[0].legend(handles=EVENT_LEGEND, fontsize=7, loc='upper right', ncol=4)
axes[0].set_xlim(0, subject['t'][-1])
axes[-1].set_xlabel('Time (s)'); plt.tight_layout(); plt.show()

## 7. PSD — Alpha Focus (1–30 Hz, 60 Hz notch)

In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(16, 20))
fig.suptitle(f'PSD After 60 Hz Notch — {subject["name"]}', fontsize=14, fontweight='bold', y=1.0)
for i in range(N_CHANNELS):
    ax = axes.flatten()[i]
    cleaned = notch_filter(eeg[i])
    f, pxx = signal.welch(cleaned, fs=FS, nperseg=4096)
    mask = (f >= 1) & (f <= 30)
    ax.plot(f[mask], pxx[mask], lw=1.5, color='#2c3e50')
    amask = (f >= 8) & (f <= 13) & mask
    ax.fill_between(f[amask], pxx[amask], alpha=0.4, color='#e67e22', label='Alpha')
    ax.set_title(CH_LABELS[i], fontsize=9, fontweight='bold')
    ax.legend(fontsize=7); ax.tick_params(labelsize=7)
axes.flatten()[11].set_visible(False)
plt.tight_layout(); plt.show()

## 8. Spectrograms — Per Channel

In [ ]:
for i in range(N_CHANNELS):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 7), gridspec_kw={'height_ratios': [1, 3]})
    fig.suptitle(f'Spectrogram — {subject["name"]} — {CH_LABELS[i]}', fontsize=12, fontweight='bold')
    cleaned = notch_filter(eeg[i])
    bp = bandpass_filter(cleaned, 1, 45)
    ax1.plot(subject['t'][::10], bp[::10], lw=0.3, color='#34495e')
    ax1.set_ylabel('µV'); ax1.set_xlim(0, subject['t'][-1])
    add_event_markers(ax1, subject['events_oc'], subject['stim_blocks'], subject['close_epochs'])
    nperseg = 2048
    f_s, t_s, Sxx = signal.spectrogram(cleaned, fs=FS, nperseg=nperseg, noverlap=nperseg//2, nfft=4096)
    fm = f_s <= 45
    im = ax2.pcolormesh(t_s, f_s[fm], 10*np.log10(Sxx[fm]+1e-10), shading='gouraud', cmap='inferno', vmin=-20)
    ax2.set_ylabel('Hz'); ax2.set_xlabel('Time (s)'); ax2.set_ylim(1, 45)
    ax2.axhline(8, color='cyan', lw=0.8, ls='--', alpha=0.7)
    ax2.axhline(13, color='cyan', lw=0.8, ls='--', alpha=0.7)
    add_event_markers(ax2, subject['events_oc'], subject['stim_blocks'], subject['close_epochs'], shade_close=False)
    plt.colorbar(im, ax=ax2, label='dB')
    plt.tight_layout(); plt.show(); print()

## 9. Band Decomposition — Per Channel

In [ ]:
for i in range(N_CHANNELS):
    fig, axs = plt.subplots(len(BANDS)+1, 1, figsize=(16, 10), sharex=True,
                             gridspec_kw={'height_ratios': [2]+[1]*len(BANDS)})
    fig.suptitle(f'Band Decomposition — {subject["name"]} — {CH_LABELS[i]}', fontsize=12, fontweight='bold')
    cleaned = notch_filter(eeg[i])
    ds = 20
    bp_full = bandpass_filter(cleaned, 1, 45)
    axs[0].plot(subject['t'][::ds], bp_full[::ds], lw=0.3, color='#2c3e50')
    axs[0].set_title('Broadband (1–45 Hz)', fontsize=10); axs[0].set_ylabel('µV', fontsize=8)
    for j, (bname, (lo, hi)) in enumerate(BANDS.items()):
        ax = axs[j+1]
        bp = bandpass_filter(cleaned, lo, hi)
        ax.plot(subject['t'][::ds], bp[::ds], lw=0.4, color=BAND_COLORS[bname])
        ax.set_title(f'{bname} ({lo}–{hi} Hz)', fontsize=10, color=BAND_COLORS[bname])
        ax.set_ylabel('µV', fontsize=8)
    for ax in axs:
        ax.set_xlim(0, subject['t'][-1]); ax.tick_params(labelsize=7)
        add_event_markers(ax, subject['events_oc'], subject['stim_blocks'], subject['close_epochs'])
    axs[-1].set_xlabel('Time (s)')
    plt.tight_layout(); plt.show(); print()

## 10. Alpha Envelope with Events

In [ ]:
fig, axes = plt.subplots(11, 1, figsize=(17, 24), sharex=True)
fig.suptitle(f'Alpha Envelope (8–13 Hz) — {subject["name"]}', fontsize=14, fontweight='bold', y=1.0)
ds = 50
for i in range(N_CHANNELS):
    ax = axes[i]; env = results['alpha_envelopes'][i]
    ax.plot(subject['t'][::ds], env[::ds], lw=1, color='#e74c3c')
    ax.fill_between(subject['t'][::ds], 0, env[::ds], alpha=0.2, color='#e74c3c')
    ax.set_ylabel(f'Ch{i+1}', fontsize=9); ax.tick_params(labelsize=8)
    add_event_markers(ax, subject['events_oc'], subject['stim_blocks'], subject['close_epochs'])
axes[0].legend(handles=EVENT_LEGEND, fontsize=7, loc='upper right', ncol=4)
axes[0].set_xlim(0, subject['t'][-1])
axes[-1].set_xlabel('Time (s)'); plt.tight_layout(); plt.show()

## 11. Eyes Open vs Closed

In [ ]:
if subject['open_epochs'] and subject['close_epochs']:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(f'Alpha Reactivity — {subject["name"]}', fontsize=13, fontweight='bold')
    x = np.arange(N_CHANNELS); w = 0.35
    ax1.bar(x-w/2, results['alpha_open'], w, color='#27ae60', label='Open', edgecolor='k', lw=0.5)
    ax1.bar(x+w/2, results['alpha_closed'], w, color='#3498db', label='Closed', edgecolor='k', lw=0.5)
    ax1.set_xticks(x); ax1.set_xticklabels(SHORT_LABELS, fontsize=9)
    ax1.set_ylabel('Alpha Power (µV²/Hz)'); ax1.set_title('Absolute'); ax1.legend(); ax1.set_yscale('log')
    ax2.bar(x, results['alpha_reactivity'], color=[ch_color(i) for i in range(N_CHANNELS)], edgecolor='k', lw=0.5)
    ax2.axhline(1, color='gray', lw=1, ls='--')
    ax2.set_xticks(x); ax2.set_xticklabels(SHORT_LABELS, fontsize=9)
    ax2.set_ylabel('Closed / Open'); ax2.set_title('Reactivity (>1 = Berger effect)')
    ax2.legend(handles=ELEC_LEGEND, fontsize=7, loc='upper right')
    plt.tight_layout(); plt.show()
else:
    print("No eyes open/close markers found for this subject.")

## 12. Visual Evoked Potential

In [ ]:
if subject['avg_data'] is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'VEP Comparison — {subject["name"]} (n={subject["avg_n_segments"] or "?"})', fontsize=13, fontweight='bold')
    avg_t = subject['avg_t']; avg_data = subject['avg_data']
    ax = axes[0]
    for idx in IDX_SW_TEEG:
        ax.plot(avg_t, avg_data[idx], lw=1.2, alpha=0.7, label=f'Ch{idx+1}')
    ax.axvline(0, color='red', lw=1, ls='--', alpha=0.5); ax.axhline(0, color='gray', lw=0.3)
    ax.set_title('Saltwater tEEG'); ax.set_xlabel('ms'); ax.set_ylabel('µV')
    ax.legend(fontsize=8); ax.set_xlim(-100, 400)
    ax = axes[1]
    ax.plot(avg_t, avg_data[9], lw=2, color='#2ecc71', label='Ch10 Paste tEEG')
    ax.plot(avg_t, avg_data[10], lw=2, color='#e74c3c', label='Ch11 Disc')
    ax.plot(avg_t, avg_data[8], lw=1.5, color='#9b59b6', ls='--', label='Ch9 Paste Conv')
    ax.axvline(0, color='red', lw=1, ls='--', alpha=0.5); ax.axhline(0, color='gray', lw=0.3)
    ax.set_title('Paste tEEG vs Disc'); ax.set_xlabel('ms'); ax.legend(fontsize=8); ax.set_xlim(-100, 400)
    ax = axes[2]
    ax.plot(avg_t, np.mean(avg_data[IDX_SW_TEEG], axis=0), lw=2, color='#3498db', label='Avg SW tEEG')
    ax.plot(avg_t, avg_data[9], lw=2, color='#2ecc71', label='Paste tEEG')
    ax.plot(avg_t, avg_data[10], lw=2, color='#e74c3c', label='Disc')
    ax.axvline(0, color='red', lw=1, ls='--', alpha=0.5); ax.axhline(0, color='gray', lw=0.3)
    ax.set_title('Grand Average by Type'); ax.set_xlabel('ms'); ax.legend(fontsize=8); ax.set_xlim(-100, 400)
    plt.tight_layout(); plt.show()
else:
    print("No averaged VEP data available for this subject.")

## 13. Disc Correlation

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(N_CHANNELS), results['disc_correlation'],
       color=[ch_color(i) for i in range(N_CHANNELS)], edgecolor='k', lw=0.5)
ax.set_xticks(range(N_CHANNELS)); ax.set_xticklabels(SHORT_LABELS)
ax.set_ylabel('Pearson r'); ax.set_ylim(0, 1.05)
ax.set_title(f'Alpha Envelope Correlation with Ch11 (Disc) — {subject["name"]}', fontsize=13, fontweight='bold')
for i, r in enumerate(results['disc_correlation']):
    ax.text(i, r + 0.02, f'{r:.3f}', ha='center', fontsize=8, fontweight='bold')
ax.legend(handles=ELEC_LEGEND, fontsize=7, loc='upper left')
plt.tight_layout(); plt.show()